In [16]:
# import libraries
import pandas as pd
import json
import re
import sys
from sklearn.metrics import classification_report as sklearn_classification_report
from seqeval.metrics import classification_report as seqeval_classification_report

# import the annotations
with open("../01_data/annotations.json", "r") as f:
    data = json.load(f)

# import the dictionary
group_dictionary_df = pd.read_csv("../01_data/groups_dictionary.csv")

In [17]:
# function to convert the annotations to bio tags on the word level
def tokenize_word_level(sentence):
    # get all words' start and end index
    word_spans = []
    char_idx = 0

    # split sentence via regex which ensures to also split at punctuation
    words = re.findall(r"\w+|[^\w\s]", sentence)

    for word in words:
        start_idx = sentence.find(word, char_idx)
        end_idx = start_idx + len(word)
        word_spans.append((start_idx, end_idx))
        char_idx = end_idx
    
    return words, word_spans

def text_to_bio(task):

    # extract sentence and annotations
    sentence = task["sentence"]
    annotations = task["annotations"]

    words, word_spans = tokenize_word_level(sentence)

    # initialize all tags as being O
    bio_tags = ["O"] * len(words)
    
    # loop through the annotations
    for annotation in annotations:
        start_ann, end_ann = annotation["start"], annotation["end"]
        for idx, (start_idx, end_idx) in enumerate(word_spans):
            if start_idx == start_ann:
                bio_tags[idx] = "B-sg"
            elif start_idx > start_ann and end_idx <= end_ann:
                bio_tags[idx] = "I-sg"

    return bio_tags

# add bio tags to the dataset
for task in data:
    bio_tags = text_to_bio(task)
    task["bio_tags"] = bio_tags

In [18]:
sys.path.append("../02_utils/")
from preprocessing import create_regex_pattern

# create the regex pattern
combined_regex = create_regex_pattern(group_dictionary_df)

In [19]:
def find_dictionary_matches(sentence, dictionary_regex):
    
    # first tokenize the sentence
    words, word_spans = tokenize_word_level(sentence)

    bio_tags = ["O"] * len(words)

    for match in re.finditer(sentence, dictionary_regex, re.IGNORECASE):
        start_match, end_match = match.span()

        for idx, (start_idx, end_idx) in enumerate(word_spans):
            if start_idx == start_match:
                bio_tags[idx] = "B-sg"
            elif start_idx > start_match and end_idx <= end_match:
                bio_tags[idx] = "I-sg"
    
    return bio_tags

In [25]:
combined_regex

"\\bemployee\\w*\\b|\\bworker\\w*\\b|\\bemployer\\w*\\b|\\bbosses\\b|\\bbusiness\\s+owners\\b|\\bbusiness\\s+people\\b|\\bself\\-employed\\b|\\bunemployed\\b|\\bthose\\s+without\\s+a\\s+job\\b|\\bthe\\s+jobless\\b|\\bjobseekers\\b|\\bpeople\\s+without\\s+employment\\b|\\blong\\-term\\s+unemployed\\b|\\bunemployed\\s+person\\b|\\bscientists\\b|\\bphysicians\\b|\\bGP\\b|\\bGPs\\b|\\bGP's\\b|\\bmanagers\\b|\\bcivil\\s+servants\\b|\\bpsychologists\\b|\\bpsychiatrists\\b|\\bdoctors\\b|\\bengineers\\b|\\bactuaries\\b|\\baccountants\\b|\\barchitects\\b|\\bsurveyors\\b|\\bgraphic\\s+designers\\b|\\bprofessors\\b|\\blecturers\\b|\\bsalesmen\\b|\\bsaleswomen\\b|\\bsalespeple\\b|\\bbusinessmen\\b|\\bbusinesswomen\\b|\\bbusiness\\s+people\\b|\\bjournalists\\b|\\blawyers\\b|\\bbarristers\\b|\\bsolicitors\\b|\\bjudges\\b|\\bveteranarians\\b|\\bpastors\\b|\\bpriests\\b|\\bartists\\b|\\bmusicians\\b|\\bbrokers\\b|\\bchief\\s+executives\\b|\\bCEOs\\b|\\bentrepreneurs\\b|\\bpublic\\s+servants\\b|\\bacad

In [30]:
# test if the function works correctly
test_sentence = "We represent working people."
matches = re.findall(test_sentence, combined_regex, re.IGNORECASE)
for match in matches:
    print(match)
#find_dictionary_matches(test_sentence, combined_regex)

In [20]:
# evaluate on the word level

# store all bio tags in a list
gt_bio = [tag for sent in data for tag in sent["bio_tags"]]

pred_bio_compound = []

for task in data:
    sentence = task["sentence"]
    pred_tags = find_dictionary_matches(sentence, combined_regex)
    pred_bio_compound.append(pred_tags)

# flatten the list
pred_bio = [tag for sent in pred_bio_compound for tag in sent]

print(sklearn_classification_report(gt_bio, pred_bio))


              precision    recall  f1-score   support

        B-sg       0.00      0.00      0.00       597
        I-sg       0.00      0.00      0.00       652
           O       0.96      1.00      0.98     28074

    accuracy                           0.96     29323
   macro avg       0.32      0.33      0.33     29323
weighted avg       0.92      0.96      0.94     29323



/Users/maxweiland/Desktop/SEDS/Master_Thesis/venv_thesis/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxweiland/Desktop/SEDS/Master_Thesis/venv_thesis/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxweiland/Desktop/SEDS/Master_Thesis/venv_thesis/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to

In [ ]:
# evaluate on the entity level

In [ ]:
# evaluate on the sentence level